# 🎤 ASR Pipeline Complet: Whosper + Confidence + Post-processing + WER
**GalsenAI Corpus Wolof | Babacar Ndao**

Étapes:
1. ✅ Chunk/Stride (20s chunks, 18s stride = 2s overlap)
2. ✅ Confidence extraction
3. ✅ Post-processing (dédup, fusion, filtrage)
4. ✅ WER/CER measurement

## 📥 CELLULE 1: Setup + Installation

In [ ]:
# Vérifier GPU
import torch
print(f"🖥️  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# Installer dépendances
!pip install -q git+https://github.com/sudoping01/whosper.git
!pip install -q librosa numpy peft transformers accelerate
print("✅ Packages installés")

## 🔹 CELLULE 2: Importer et définir classes

In [ ]:
import json
import numpy as np
import librosa
from pathlib import Path
from typing import List, Dict
import torch
from transformers import AutoModelForSpeech2Seq2Seq, AutoProcessor
from peft import PeftModel

# ==================== ÉTAPE 1: CHUNK & STRIDE ====================

class AudioChunker:
    """Découpe l'audio avec overlap configurable."""
    
    def __init__(self, chunk_duration: int = 20, stride: int = 18):
        """
        Args:
            chunk_duration: durée du chunk en secondes (défaut 20s, était 12s)
            stride: distance entre starts de chunks (défaut 18s → 2s overlap)
        """
        self.chunk_duration = chunk_duration
        self.stride = stride
        self.overlap = chunk_duration - stride
        
    def chunk_audio(self, audio: np.ndarray, sr: int = 16000) -> List[np.ndarray]:
        """Découpe l'audio en chunks avec overlap."""
        chunk_samples = self.chunk_duration * sr
        stride_samples = self.stride * sr
        
        chunks = []
        for start in range(0, len(audio), stride_samples):
            end = start + chunk_samples
            if end > len(audio):
                chunk = np.zeros(chunk_samples)
                chunk[:len(audio)-start] = audio[start:len(audio)]
            else:
                chunk = audio[start:end]
            chunks.append(chunk)
        
        return chunks
    
    def get_chunk_metadata(self) -> Dict:
        """Retourne config pour logs."""
        return {
            "chunk_duration_s": self.chunk_duration,
            "stride_s": self.stride,
            "overlap_s": self.overlap,
            "overlap_percentage": (self.overlap / self.chunk_duration) * 100
        }

print("✅ AudioChunker défini")

In [ ]:
# ==================== ÉTAPE 2: CONFIDENCE EXTRACTION ====================

class WhosperTranscriber:
    """Whosper-large-v2 avec extraction confidence scores."""
    
    def __init__(self, model_id: str = "CAYTU/whosper-large-v2"):
        """Charge le modèle Whosper avec PEFT adapter."""
        print(f"📥 Chargement {model_id}...")
        
        base_model_id = "openai/whisper-large-v2"
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        
        self.model = AutoModelForSpeech2Seq2Seq.from_pretrained(
            base_model_id,
            torch_dtype=dtype,
            low_cpu_mem_usage=True,
            use_safetensors=True
        ).to(device)
        
        # Charger PEFT adapter
        self.model = PeftModel.from_pretrained(self.model, model_id)
        self.model = self.model.to(device)
        
        self.processor = AutoProcessor.from_pretrained(base_model_id)
        self.device = device
        self.dtype = dtype
        
        print(f"✅ Modèle chargé sur {device}")
    
    def transcribe_with_confidence(self, audio: np.ndarray, sr: int = 16000) -> Dict:
        """Transcrit et extrait confidence score."""
        inputs = self.processor(
            audio,
            sampling_rate=sr,
            return_tensors="pt",
            return_attention_mask=True
        ).to(self.device)
        
        input_features = inputs.input_features.to(self.dtype)
        attention_mask = inputs.get("attention_mask")
        
        with torch.no_grad():
            outputs = self.model.generate(
                inputs=input_features,
                attention_mask=attention_mask,
                max_new_tokens=300,
                task="transcribe",
                return_dict_in_generate=True,
                output_scores=True
            )
        
        text = self.processor.batch_decode(outputs.sequences, skip_special_tokens=True)[0]
        
        # Confidence = moyenne log-probs
        if outputs.scores:
            log_probs = []
            for i, scores in enumerate(outputs.scores):
                token_logprobs = torch.log_softmax(scores, dim=-1)
                top_logprob = torch.max(token_logprobs, dim=-1).values[0]
                log_probs.append(top_logprob.item())
            
            confidence = np.exp(np.mean(log_probs))
            confidence = float(np.clip(confidence, 0, 1))
        else:
            confidence = 0.5
        
        return {
            "text": text.strip(),
            "confidence": confidence,
            "log_probs": log_probs if outputs.scores else []
        }

print("✅ WhosperTranscriber défini")

In [ ]:
# ==================== ÉTAPE 3: POST-PROCESSING ====================

class PostProcessor:
    """Nettoyage et fusion des transcriptions."""
    
    @staticmethod
    def deduplicate_repeats(text: str, min_repeat_count: int = 3) -> str:
        """Supprime les répétitions de mots/phrases."""
        words = text.split()
        cleaned = []
        i = 0
        
        while i < len(words):
            word = words[i]
            repeat_count = 1
            while i + repeat_count < len(words) and words[i + repeat_count] == word:
                repeat_count += 1
            
            if repeat_count < min_repeat_count:
                cleaned.extend([word] * repeat_count)
            else:
                cleaned.append(word)
            
            i += repeat_count
        
        return " ".join(cleaned)
    
    @staticmethod
    def filter_by_confidence(transcriptions: List[Dict], threshold: float = 0.5) -> List[Dict]:
        """Filtre les chunks avec confidence < threshold."""
        filtered = []
        low_conf_count = 0
        
        for trans in transcriptions:
            if trans["confidence"] >= threshold:
                filtered.append(trans)
            else:
                low_conf_count += 1
        
        print(f"   ⚠️  {low_conf_count} chunks avec confidence < {threshold}")
        return filtered
    
    @staticmethod
    def merge_overlapping_chunks(chunks: List[Dict], overlap_duration: int = 2) -> str:
        """Fusionne chunks chevauchants en détectant duplication."""
        if not chunks:
            return ""
        
        merged_text = chunks[0]["text"]
        
        for i in range(1, len(chunks)):
            current_text = chunks[i]["text"]
            prev_words = merged_text.split()
            curr_words = current_text.split()
            
            overlap_found = False
            for overlap_len in range(min(5, len(prev_words), len(curr_words)), 0, -1):
                if prev_words[-overlap_len:] == curr_words[:overlap_len]:
                    merged_text += " " + " ".join(curr_words[overlap_len:])
                    overlap_found = True
                    break
            
            if not overlap_found:
                merged_text += " " + current_text
        
        return merged_text.strip()

print("✅ PostProcessor défini")

In [ ]:
# ==================== ÉTAPE 4: WER MEASUREMENT ====================

class WERCalculator:
    """Calcule Word Error Rate (WER) entre référence et hypothèse."""
    
    @staticmethod
    def compute_wer(reference: str, hypothesis: str) -> float:
        """Calcule WER = (S + D + I) / N"""
        ref_words = reference.split()
        hyp_words = hypothesis.split()
        
        d = np.zeros((len(ref_words) + 1, len(hyp_words) + 1))
        
        for i in range(len(ref_words) + 1):
            d[i][0] = i
        for j in range(len(hyp_words) + 1):
            d[0][j] = j
        
        for i in range(1, len(ref_words) + 1):
            for j in range(1, len(hyp_words) + 1):
                if ref_words[i-1] == hyp_words[j-1]:
                    d[i][j] = d[i-1][j-1]
                else:
                    d[i][j] = 1 + min(
                        d[i-1][j],
                        d[i][j-1],
                        d[i-1][j-1]
                    )
        
        wer = d[len(ref_words)][len(hyp_words)] / len(ref_words) if len(ref_words) > 0 else 0
        return float(wer)
    
    @staticmethod
    def compute_cer(reference: str, hypothesis: str) -> float:
        """Character Error Rate (CER)"""
        ref_chars = list(reference.replace(" ", ""))
        hyp_chars = list(hypothesis.replace(" ", ""))
        
        d = np.zeros((len(ref_chars) + 1, len(hyp_chars) + 1))
        for i in range(len(ref_chars) + 1):
            d[i][0] = i
        for j in range(len(hyp_chars) + 1):
            d[0][j] = j
        
        for i in range(1, len(ref_chars) + 1):
            for j in range(1, len(hyp_chars) + 1):
                if ref_chars[i-1] == hyp_chars[j-1]:
                    d[i][j] = d[i-1][j-1]
                else:
                    d[i][j] = 1 + min(d[i-1][j], d[i][j-1], d[i-1][j-1])
        
        cer = d[len(ref_chars)][len(hyp_chars)] / len(ref_chars) if len(ref_chars) > 0 else 0
        return float(cer)

print("✅ WERCalculator défini")

## 🚀 CELLULE 3: Exécuter le pipeline complet

In [ ]:
# ==================== CONFIG ====================

# Remplace par ton chemin vidéo Colab
AUDIO_PATH = "/tmp/wolof_videos/_eqpwNEXEzs.mp4"  # ou "vrZIxImDf8Y.mp4"
OUTPUT_DIR = "/tmp/wolof_transcriptions_v2"

# Paramètres pipeline
CHUNK_DURATION = 20  # 20s (était 12s) → plus de contexte
STRIDE = 18          # 18s → 2s overlap (était 11s → 1s)
CONFIDENCE_THRESHOLD = 0.5  # Filtrer confidence < 50%

print(f"📁 Fichier: {AUDIO_PATH}")
print(f"⚙️  Config: chunk={CHUNK_DURATION}s, stride={STRIDE}s, overlap={(CHUNK_DURATION-STRIDE)}s")

In [ ]:
# Étape 1: Chargement audio + chunking
print(f"\n{'='*70}")
print(f"🎤 STEP 1: CHUNKING")
print(f"{'='*70}")

audio, sr = librosa.load(AUDIO_PATH, sr=16000)
duration_s = len(audio) / sr

chunker = AudioChunker(chunk_duration=CHUNK_DURATION, stride=STRIDE)
chunks = chunker.chunk_audio(audio, sr=sr)
metadata = chunker.get_chunk_metadata()

print(f"\n✅ Audio chargé:")
print(f"   Durée: {duration_s:.1f}s")
print(f"   Chunks: {len(chunks)}")
print(f"   Config: {metadata}")

In [ ]:
# Étape 2: Transcription + Confidence
print(f"\n{'='*70}")
print(f"🎤 STEP 2: TRANSCRIPTION + CONFIDENCE EXTRACTION")
print(f"{'='*70}")

transcriber = WhosperTranscriber()
all_transcriptions = []

for i, chunk in enumerate(chunks):
    result = transcriber.transcribe_with_confidence(chunk, sr=sr)
    all_transcriptions.append({
        "chunk_id": i,
        **result
    })
    
    if (i + 1) % 50 == 0 or i == len(chunks) - 1:
        avg_conf = np.mean([t["confidence"] for t in all_transcriptions])
        print(f"   [{i+1:3d}/{len(chunks)}] Confidence moyenne: {avg_conf:.3f}")

print(f"\n✅ {len(all_transcriptions)} chunks transcrits")

In [ ]:
# Étape 3: Post-processing
print(f"\n{'='*70}")
print(f"🔧 STEP 3: POST-PROCESSING")
print(f"{'='*70}")

# 3a. Filtrer par confidence
print(f"\n3a. FILTRAGE PAR CONFIDENCE")
print(f"   Avant: {len(all_transcriptions)} chunks")
filtered = PostProcessor.filter_by_confidence(all_transcriptions, threshold=CONFIDENCE_THRESHOLD)
print(f"   Après: {len(filtered)} chunks")

# 3b. Merger
print(f"\n3b. FUSION CHUNKS")
merged_text = PostProcessor.merge_overlapping_chunks(filtered)
print(f"   Caractères: {len(merged_text)}")

# 3c. Dédupliquer
print(f"\n3c. DÉDUPLICATION")
before_dedup = len(merged_text)
deduplicated = PostProcessor.deduplicate_repeats(merged_text)
print(f"   Avant: {before_dedup} chars")
print(f"   Après: {len(deduplicated)} chars")
print(f"   Réduction: {100 * (before_dedup - len(deduplicated)) / before_dedup:.1f}%")

print(f"\n✅ Post-processing complet")

In [ ]:
# Étape 4: Export JSON + WER (optionnel)
print(f"\n{'='*70}")
print(f"📊 STEP 4: EXPORT + WER MEASUREMENT")
print(f"{'='*70}")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Préparer output
output_data = {
    "video_id": Path(AUDIO_PATH).stem,
    "duration_s": duration_s,
    "num_chunks": len(chunks),
    "chunk_config": metadata,
    "transcription_raw": merged_text,
    "transcription_cleaned": deduplicated,
    "quality_metrics": {
        "avg_confidence": round(np.mean([t["confidence"] for t in filtered]), 3),
        "chunks_low_confidence": len(all_transcriptions) - len(filtered),
        "total_characters": len(deduplicated)
    }
}

# Sauvegarder
output_file = Path(OUTPUT_DIR) / f"{Path(AUDIO_PATH).stem}_v2_asr.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"\n📁 Sauvegardé: {output_file}")
print(f"\n✅ RÉSUMÉ:")
print(f"   Durée vidéo: {duration_s:.1f}s")
print(f"   Chunks traités: {len(chunks)}")
print(f"   Chunks valides (conf>{CONFIDENCE_THRESHOLD}): {len(filtered)}")
print(f"   Caractères finaux: {len(deduplicated)}")
print(f"   Confidence moyenne: {output_data['quality_metrics']['avg_confidence']:.3f}")

## 📈 CELLULE 4: Optionnel - WER avec référence Wolof

In [ ]:
# Si tu as une référence transcription Wolof, la tester ici

# Exemple: première phrase de vidéo 1
reference_1 = "ndeysaan coyy ati na Cëy àddina àddina day dox jamono day dara"

# Récupérer première phrase de ton output
hypothesis = deduplicated[:150]  # First 150 chars

print(f"\n{'='*70}")
print(f"📊 WER/CER COMPARISON")
print(f"{'='*70}")
print(f"\nRéférence: {reference_1[:80]}...")
print(f"Hypothèse: {hypothesis[:80]}...")

wer = WERCalculator.compute_wer(reference_1, hypothesis)
cer = WERCalculator.compute_cer(reference_1, hypothesis)

print(f"\n✅ SCORES:")
print(f"   WER (Word Error Rate): {wer:.1%}")
print(f"   CER (Char Error Rate): {cer:.1%}")

if wer < 0.25:
    print(f"   ✅ BON (WER < 25%)")
elif wer < 0.35:
    print(f"   ⚠️  MOYEN (WER 25-35%)")
else:
    print(f"   ❌ FAIBLE (WER > 35%)")

## 💡 CELLULE 5: Aperçu du texte final

In [ ]:
# Afficher les premières lignes du texte transcrit
print(f"\n{'='*70}")
print(f"📝 APERÇU TRANSCRIPTION (premiers 500 chars)")
print(f"{'='*70}\n")
print(deduplicated[:500])
print(f"\n... ({len(deduplicated) - 500} caractères restants)")

## 📋 Notes pour amélioration

**Prochaines étapes si WER encore élevé:**

1. **Augmenter overlap davantage** (stride=15-16s → 4-5s overlap)
2. **Fine-tuner Whosper sur Wolof** (nécessite 500h+ data annotée)
3. **Post-processing linguistique**:
   - Règles Wolof (accords, conjugaisons)
   - Language model pour correction contexte
4. **Confidence filtering plus agressif** (threshold=0.6-0.7)
5. **Essayer modèles alternatifs** (whisper-medium, wav2vec2-large-xlsr)

**Pour corpus GalsenAI:**
- ✅ Utiliser cette pipeline pour transcription brute
- ✅ Export JSON avec confidence scores par chunk
- ✅ Validation humaine (linguistes Wolof) nécessaire
- ✅ Itérer: feedback → réentrainement